# Review copy

Historical hand-tracker experiments, original cells 0–14. Extracted from the Drive notebook experiments_and _tests.ipynb without changing code or Markdown. Outputs and execution counts were removed to reduce file size. This part is for review and is not standalone. Original cell order is preserved.

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:

from google.colab import drive
import shutil
import zipfile
import os

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/reinterhand_5114_dataset.zip'
dest_dir = '/content/dataset'

os.makedirs(dest_dir, exist_ok=True)
print("Extracting dataset to local Colab SSD...")
!unzip -q {zip_path} -d {dest_dir}
print("Extraction complete!")

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

# 1. Environment & Device Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Accelerator      : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found at {MANIFEST_PATH}. Check extraction output.")

# 2. File Path Resolution Helper
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

# 3. Read Manifest
records = []
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

print(f"Loaded manifest records: {len(records):,} samples")

# 4. In-Memory Image Pre-caching (256x192 uint8 RGB)
TARGET_W, TARGET_H = 256, 192

cached_rgb = []
cached_targets = []
valid_records = []

print("\nPre-caching 5,114 resized images and 3D targets into RAM...")
start_time = time.time()

for row in tqdm(records, desc="Ingesting Samples"):
    try:
        img_p = resolve_file(DATA_DIR, row["image_path"], "images")
        kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

        # Read & resize RGB image
        img_bgr = cv2.imread(str(img_p))
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        # Read 3D landmarks (24 joints x 3, in millimeters)
        with open(kp_p, "r", encoding="utf-8") as f:
            kps = np.array(json.load(f), dtype=np.float32)

        # Compute root-relative coordinate target (wrist = joint 0)
        kps_rel = (kps - kps[0:1]).flatten()

        cached_rgb.append(img_resized)
        cached_targets.append(kps_rel)
        valid_records.append(row)
    except Exception as e:
        continue

cached_rgb = np.stack(cached_rgb, axis=0)                     # (N, 192, 256, 3) uint8
cached_targets = torch.tensor(np.stack(cached_targets), dtype=torch.float32)  # (N, 72) float32

elapsed = time.time() - start_time
ram_usage_mb = (cached_rgb.nbytes + cached_targets.untyped_storage().nbytes()) / (1024 * 1024)
print(f"Cached {len(cached_rgb):,} samples in {elapsed:.1f}s (RAM Footprint: {ram_usage_mb:.1f} MB)")

# 5. Segment-Stratified Split (80% Train, 20% Val per interaction)
segment_indices = defaultdict(list)
for idx, r in enumerate(valid_records):
    segment_indices[r["segment"]].append(idx)

train_indices, val_indices = [], []
for seg, idxs in segment_indices.items():
    n = len(idxs)
    n_train = max(1, int(0.8 * n))
    train_indices.extend(idxs[:n_train])
    val_indices.extend(idxs[n_train:])

train_indices = np.array(train_indices)
val_indices = np.array(val_indices)
print(f"Train samples: {len(train_indices):,} | Validation samples: {len(val_indices):,} across {len(segment_indices)} segments")

# 6. Representation Generator
def compute_representation(img_rgb: np.ndarray, rep_key: str) -> np.ndarray:
    img_f = img_rgb.astype(np.float32)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)

    if rep_key == "rgb":
        return (img_f / 255.0).transpose(2, 0, 1)

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3) / 1020.0
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3) / 1020.0
    g_mag = np.sqrt(gx**2 + gy**2)

    if rep_key == "sobel_mag":
        return g_mag[np.newaxis, ...]
    elif rep_key == "sobel_dir":
        return np.stack([gx, gy, g_mag], axis=0)

    # Multi-spectral Di Zenzo structure tensor
    rx = cv2.Sobel(img_f[:, :, 0], cv2.CV_32F, 1, 0, ksize=3)
    ry = cv2.Sobel(img_f[:, :, 0], cv2.CV_32F, 0, 1, ksize=3)
    gx_c = cv2.Sobel(img_f[:, :, 1], cv2.CV_32F, 1, 0, ksize=3)
    gy_c = cv2.Sobel(img_f[:, :, 1], cv2.CV_32F, 0, 1, ksize=3)
    bx = cv2.Sobel(img_f[:, :, 2], cv2.CV_32F, 1, 0, ksize=3)
    by = cv2.Sobel(img_f[:, :, 2], cv2.CV_32F, 0, 1, ksize=3)

    gxx = rx**2 + gx_c**2 + bx**2
    gyy = ry**2 + gy_c**2 + by**2
    gxy = rx * ry + gx_c * gy_c + bx * by

    diff = gxx - gyy
    disc = np.sqrt(np.maximum(diff**2 + 4.0 * (gxy**2), 0.0))
    lambda_max = 0.5 * ((gxx + gyy) + disc)
    dizenzo_mag = np.sqrt(np.maximum(lambda_max, 0.0)) / 1766.7

    if rep_key == "dizenzo":
        return dizenzo_mag[np.newaxis, ...]
    elif rep_key == "hybrid":
        return np.stack([gx, gy, g_mag, dizenzo_mag], axis=0)

# 7. Lightweight Pose Regression Network
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int, out_dim: int = 72):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# 8. Training & Benchmarking Loop
VARIANTS = [
    ("Scalar Sobel (|G|)", "sobel_mag", 1),
    ("Directional Sobel (Gx, Gy, |G|)", "sobel_dir", 3),
    ("Di Zenzo Tensor (λ_max^0.5)", "dizenzo", 1),
    ("Hybrid Vector Field (Sobel + DiZenzo)", "hybrid", 4),
    ("RGB Reference Baseline", "rgb", 3)
]

benchmark_results = {}
EPOCHS = 15
BATCH_SIZE = 64

print("\n" + "=" * 80)
print(" RUNNING 5-REPRESENTATION BENCHMARK (5,114 SAMPLES)")
print("=" * 80)

for name, rep_key, channels in VARIANTS:
    print(f"\n--- Model: {name} ({channels} Ch) ---")

    # Generate representation on the fly for cached images
    rep_tensors = np.stack([compute_representation(img, rep_key) for img in cached_rgb], axis=0)
    rep_tensors = torch.from_numpy(rep_tensors).float()

    train_ds = TensorDataset(rep_tensors[train_indices], cached_targets[train_indices])
    val_ds = TensorDataset(rep_tensors[val_indices], cached_targets[val_indices])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available())

    model = PoseBackbone(in_channels=channels, out_dim=72).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_val_mpjpe = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation MPJPE computation
        model.eval()
        total_error = 0.0
        total_joints = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                pred = model(bx).view(-1, 24, 3)
                gt = by.view(-1, 24, 3)
                dist = torch.norm(pred - gt, dim=-1)  # Euclidean error per joint (mm)
                total_error += dist.sum().item()
                total_joints += dist.numel()

        val_mpjpe = total_error / total_joints
        if val_mpjpe < best_val_mpjpe:
            best_val_mpjpe = val_mpjpe

        if epoch % 5 == 0 or epoch == EPOCHS:
            avg_train = train_loss / len(train_loader)
            print(f"  Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {avg_train:8.2f} | Val MPJPE: {val_mpjpe:6.2f} mm")

    benchmark_results[name] = best_val_mpjpe

    # Free intermediate representation tensors before next iteration
    del rep_tensors, train_ds, val_ds, train_loader, val_loader, model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 9. Final Benchmark Summary
print("\n" + "=" * 80)
print(" BENCHMARK RESULTS SUMMARY: 3D POSE RECONSTRUCTION ERROR")
print("=" * 80)
print(f"{'Input Representation':<40} | {'Best Validation MPJPE':<22}")
print("-" * 80)
for name, mpjpe in sorted(benchmark_results.items(), key=lambda x: x[1]):
    print(f"{name:<40} | {mpjpe:6.2f} mm")
print("=" * 80)

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# -------------------------------------------------------------------------
# 1. HARDWARE VERIFICATION & VRAM ALLOCATION
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Hardware Accelerator : {gpu_name}")
print(f"Available VRAM       : {total_vram_gb:.2f} GB")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# Official Re:InterHand 24 -> Kinematic 21 Joint Index Mapping
# Dropping Thumb_0 (1), Pinky_0 (18), and Forearm_Stub (23)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

# -------------------------------------------------------------------------
# 2. HIGH-PERFORMANCE GPU EDGE / STRUCTURE TENSOR EXTRACTOR
# -------------------------------------------------------------------------
class GPURepresentationEngine(nn.Module):
    """Executes Sobel and Di Zenzo tensor fields entirely in CUDA VRAM."""
    def __init__(self, device: torch.device):
        super().__init__()
        # Sobel spatial differentiation kernels
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)

        self.kx_1 = kx.view(1, 1, 3, 3)
        self.ky_1 = ky.view(1, 1, 3, 3)
        self.kx_3 = kx.view(1, 1, 3, 3).repeat(3, 1, 1, 1)
        self.ky_3 = ky.view(1, 1, 3, 3).repeat(3, 1, 1, 1)

    @torch.no_grad()
    def rgb_to_gray(self, rgb_f: torch.Tensor) -> torch.Tensor:
        # Standard luminance formula (B, 1, H, W)
        return 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, rep_key: str) -> torch.Tensor:
        """
        Input: rgb_f in range [0.0, 255.0] on CUDA (B, 3, H, W)
        Output: Normalized feature tensor on CUDA
        """
        if rep_key == "rgb":
            return rgb_f / 255.0

        gray = self.rgb_to_gray(rgb_f)

        if rep_key in ("sobel_mag", "sobel_dir", "hybrid"):
            gx = F.conv2d(gray, self.kx_1, padding=1) / 1020.0
            gy = F.conv2d(gray, self.ky_1, padding=1) / 1020.0
            g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)

            if rep_key == "sobel_mag":
                return g_mag
            elif rep_key == "sobel_dir":
                return torch.cat([gx, gy, g_mag], dim=1)

        if rep_key in ("dizenzo", "hybrid"):
            # Multi-channel color gradient via grouped convolution
            grad_x = F.conv2d(rgb_f, self.kx_3, padding=1, groups=3)
            grad_y = F.conv2d(rgb_f, self.ky_3, padding=1, groups=3)
            rx, gx_c, bx = grad_x[:, 0:1], grad_x[:, 1:2], grad_x[:, 2:3]
            ry, gy_c, by = grad_y[:, 0:1], grad_y[:, 1:2], grad_y[:, 2:3]

            gxx = rx**2 + gx_c**2 + bx**2
            gyy = ry**2 + gy_c**2 + by**2
            gxy = rx * ry + gx_c * gy_c + bx * by

            diff = gxx - gyy
            disc = torch.sqrt(torch.clamp(diff**2 + 4.0 * (gxy**2), min=0.0))
            lambda_max = 0.5 * ((gxx + gyy) + disc)
            dizenzo_mag = torch.sqrt(torch.clamp(lambda_max, min=0.0)) / 1766.7

            if rep_key == "dizenzo":
                return dizenzo_mag
            elif rep_key == "hybrid":
                return torch.cat([gx, gy, g_mag, dizenzo_mag], dim=1)

gpu_extractor = GPURepresentationEngine(DEVICE)

# -------------------------------------------------------------------------
# 3. FAST VRAM DATASET INGESTION (UINT8 COMPACT STORAGE)
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path}")

print("\n" + "=" * 75)
print(" INGESTING 5,114 SAMPLES DIRECTLY INTO GPU VRAM (AS UINT8)")
print("=" * 75)

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    records = list(csv.DictReader(f))

raw_images_list = []
raw_targets_list = []
sample_segments = []

for row in tqdm(records, desc="Decoding Images to RAM"):
    try:
        img_p = resolve_file(DATA_DIR, row["image_path"], "images")
        kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

        bgr = cv2.imread(str(img_p))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

        with open(kp_p, "r", encoding="utf-8") as kf:
            full_kps = np.array(json.load(kf), dtype=np.float32)

        kps_21 = full_kps[KEEP_21_INDICES]
        kps_rel = (kps_21 - kps_21[0:1]).flatten()

        # Channel-first uint8 array: (3, 192, 256)
        raw_images_list.append(rgb_resized.transpose(2, 0, 1))
        raw_targets_list.append(kps_rel)
        sample_segments.append(row["segment"])
    except Exception:
        continue

# Convert directly to GPU resident tensors
print("Transferring dataset into Tesla T4 VRAM...")
vram_images = torch.tensor(np.stack(raw_images_list), dtype=torch.uint8, device=DEVICE)
vram_targets = torch.tensor(np.stack(raw_targets_list), dtype=torch.float32, device=DEVICE)
sample_segments = np.array(sample_segments)
unique_segments = np.unique(sample_segments)

# Explicitly free CPU memory
del raw_images_list, raw_targets_list
gc.collect()

vram_used_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
print(f"Successfully staged {len(vram_images):,} frames on GPU!")
print(f"Dataset VRAM Footprint: {vram_used_mb:.1f} MB (Only {vram_used_mb / (total_vram_gb * 1024) * 100:.1f}% of total VRAM)")
print(f"Host System RAM is safe: Zero Risk of Crash.\n")

# -------------------------------------------------------------------------
# 4. LIGHTWEIGHT POSE REGRESSION ARCHITECTURE
# -------------------------------------------------------------------------
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 5. EXECUTE FULL 3-SEED SEGMENT-DISJOINT BENCHMARK (100% ON GPU)
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

VARIANTS = [
    ("Scalar Sobel (|G|)", 1, "sobel_mag"),
    ("Directional Sobel (Gx, Gy, |G|)", 3, "sobel_dir"),
    ("Di Zenzo (λ_max^0.5)", 1, "dizenzo"),
    ("Hybrid Vector Field", 4, "hybrid"),
    ("RGB Baseline", 3, "rgb")
]

experiment_matrix = {name: [] for name, _, _ in VARIANTS}

print("=" * 80)
print(f" RUNNING SEGMENT-LEVEL BENCHMARK ({len(SEEDS)} SEEDS, 5 VARIANTS)")
print("=" * 80)

for name, in_ch, rep_key in VARIANTS:
    print(f"\n>>> Representation: {name} ({in_ch} Channels) <<<")

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Segment-disjoint partition (80% train segments, 20% validation segments)
        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=in_ch, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.MSELoss()

        best_val_mpjpe = float("inf")

        for epoch in range(1, EPOCHS + 1):
            # Train step (On-the-fly GPU edge computation)
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb_batch = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx]

                x_batch = gpu_extractor(raw_rgb_batch, rep_key)

                optimizer.zero_grad()
                pred = model(x_batch)
                loss = criterion(pred, y_batch)
                loss.backward()
                optimizer.step()

            # Validation step (All inside VRAM)
            model.eval()
            total_err, total_pts = 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb_batch = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx]

                    x_batch = gpu_extractor(raw_rgb_batch, rep_key)

                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)
                    gt = y_batch.view(-1, NUM_JOINTS, 3)
                    dist = torch.norm(pred - gt, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

            val_mpjpe = total_err / total_pts
            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe

        experiment_matrix[name].append(best_val_mpjpe)
        print(f"  [Seed {seed:<3}] Train Segs: {len(train_segs):2d} | Val Segs: {len(val_segs):2d} | Best Val MPJPE: {best_val_mpjpe:6.2f} mm")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. FINAL RESULTS
# -------------------------------------------------------------------------
print("\n" + "=" * 85)
print(" RIGOROUS BENCHMARK RESULTS: SEGMENT-HELD-OUT GENERALIZATION (21 JOINTS)")
print("=" * 85)
print(f"{'Input Representation':<32} | {'Seed Runs (mm)':<22} | {'Mean ± Std MPJPE':<18}")
print("-" * 85)

for name, _, _ in VARIANTS:
    runs = experiment_matrix[name]
    mean_val = np.mean(runs)
    std_val = np.std(runs)
    runs_str = ", ".join([f"{v:.2f}" for v in runs])
    print(f"{name:<32} | {runs_str:<22} | {mean_val:6.2f} ± {std_val:4.2f} mm")

print("=" * 85)

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE VERIFICATION & VRAM PREPARATION
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Hardware Accelerator : {gpu_name}")
print(f"Available VRAM       : {total_vram_gb:.2f} GB")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure the dataset is extracted.")

# Official Re:InterHand 24 -> Kinematic 21 Joint Index Mapping
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

# -------------------------------------------------------------------------
# 2. GPU SOBEL EXTRACTOR
# -------------------------------------------------------------------------
class GPURepresentationEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx_1 = kx.view(1, 1, 3, 3)
        self.ky_1 = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        # Standard luminance (B, 1, H, W)
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx_1, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky_1, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPURepresentationEngine(DEVICE)

# -------------------------------------------------------------------------
# 3. SELF-HEALING VRAM INGESTION (SAFE UINT8 CACHING)
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("\nPreloading dataset into Tesla T4 VRAM (uint8 compact storage)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_images_list, raw_targets_list, sample_segments_list = [], [], []

    for row in records:
        try:
            img_p = resolve_file(DATA_DIR, row["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_images_list.append(rgb_resized.transpose(2, 0, 1))
            raw_targets_list.append(kps_rel)
            sample_segments_list.append(row["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_images_list), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targets_list), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(sample_segments_list)
    unique_segments = np.unique(sample_segments)

    del raw_images_list, raw_targets_list, sample_segments_list
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} existing frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 4. KINEMATIC TOPOLOGY & STRUCTURAL LOSS DEFINITIONS
# -------------------------------------------------------------------------
# 20 directed bone parent-child pairs in 21-joint space
BONE_PAIRS = [
    # Thumb: Wrist -> CMC -> MCP -> IP -> Tip
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 17), (17, 18), (18, 19), (19, 20)
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

class KinematicStructureLoss(nn.Module):
    def __init__(self, lambda_len: float = 0.5, lambda_dir: float = 10.0):
        super().__init__()
        self.lambda_len = lambda_len
        self.lambda_dir = lambda_dir

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        # Base Coordinate Position Loss (MSE)
        l_pos = F.mse_loss(pred, target)

        # Bone vectors (B, 20, 3)
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # Bone length consistency (L1 error in mm)
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # Bone direction angular alignment (Cosine distance)
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        l_dir = (1.0 - (u_pred * u_gt).sum(dim=-1)).mean()

        total = l_pos + self.lambda_len * l_len + self.lambda_dir * l_dir
        return total

# -------------------------------------------------------------------------
# 5. POSE BACKBONE (1-CHANNEL INPUT)
# -------------------------------------------------------------------------
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS: UNCONSTRAINED MSE VS. KINEMATIC LOSS
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

EXPERIMENTS = [
    ("Unconstrained MSE Loss", False),
    ("Kinematic Structural Loss", True)
]

comparison_matrix = {name: [] for name, _ in EXPERIMENTS}
bone_error_matrix = {name: [] for name, _ in EXPERIMENTS}

print("\n" + "=" * 80)
print(" BENCHMARKING LOSS OBJECTIVES ACROSS 3 SEGMENT SPLITS")
print("=" * 80)

for exp_name, use_kinematic in EXPERIMENTS:
    print(f"\n>>> Running: {exp_name} <<<")
    kin_loss_fn = KinematicStructureLoss()

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Strictly identical segment splits across both runs
        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=1, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_bone_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb_batch = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb_batch)

                optimizer.zero_grad()
                pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                loss = kin_loss_fn(pred, y_batch) if use_kinematic else F.mse_loss(pred, y_batch)
                loss.backward()
                optimizer.step()

            # Validation evaluation
            model.eval()
            total_err, total_bone_err, total_pts = 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb_batch = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb_batch)
                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                    # MPJPE (mm)
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone length deformation error (mm)
                    b_pred = torch.norm(pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES], dim=-1)
                    b_gt = torch.norm(y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES], dim=-1)
                    total_bone_err += torch.abs(b_pred - b_gt).sum().item()

            val_mpjpe = total_err / total_pts
            val_bone_err = total_bone_err / (len(val_indices) * len(BONE_PAIRS))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_bone_err = val_bone_err

        comparison_matrix[exp_name].append(best_val_mpjpe)
        bone_error_matrix[exp_name].append(best_val_bone_err)
        print(f"  [Seed {seed:<3}] Val MPJPE: {best_val_mpjpe:6.2f} mm | Bone Length Error: {best_val_bone_err:5.2f} mm")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. RESULTS SUMMARY
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" LOSS OBJECTIVE COMPARISON: 3-SPLIT SEGMENT-HELD-OUT EVALUATION")
print("=" * 80)
print(f"{'Objective Formulation':<30} | {'Mean MPJPE':<18} | {'Bone Length Error':<18}")
print("-" * 80)
for exp_name, _ in EXPERIMENTS:
    m_pjpe = np.mean(comparison_matrix[exp_name])
    s_pjpe = np.std(comparison_matrix[exp_name])
    m_bone = np.mean(bone_error_matrix[exp_name])
    s_bone = np.std(bone_error_matrix[exp_name])
    print(f"{exp_name:<30} | {m_pjpe:6.2f} ± {s_pjpe:4.2f} mm | {m_bone:5.2f} ± {s_bone:4.2f} mm")
print("=" * 80)

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# 21-Joint Topology & 20 Directed Bones
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

# -------------------------------------------------------------------------
# 1. GPU EDGE EXTRACTOR & VRAM INGESTION
# -------------------------------------------------------------------------
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into VRAM...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()
    print(f"Staged {len(vram_images):,} frames into VRAM.")
else:
    print(f"Reusing {len(vram_images):,} frames in VRAM.")

# -------------------------------------------------------------------------
# 2. CALIBRATED LOSS MODULE & NETWORK ARCHITECTURE
# -------------------------------------------------------------------------
class LossEvaluator(nn.Module):
    def __init__(self, lambda_len: float = 0.0, lambda_dir: float = 0.0):
        super().__init__()
        self.lambda_len = lambda_len
        self.lambda_dir = lambda_dir

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        # 1. Coordinate Euclidean distance in mm
        l_pos = torch.norm(pred - target, dim=-1).mean()

        # 2. Bone vectors
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # 3. Bone length L1 in mm
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # 4. Bone direction cosine distance
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        l_dir = (1.0 - (u_pred * u_gt).sum(dim=-1)).mean()

        total = l_pos + self.lambda_len * l_len + self.lambda_dir * l_dir
        return total, l_pos.item(), l_len.item(), l_dir.item()

class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 3. CONTROLLED ABLATION EXPERIMENTS
# -------------------------------------------------------------------------
ABLATION_CONFIGS = [
    ("Pos Only (mm L1)",            0.0,  0.0),
    ("Pos + Len (λ=0.2)",           0.2,  0.0),
    ("Pos + Len (λ=0.5)",           0.5,  0.0),
    ("Pos + Dir (λ=20.0)",          0.0, 20.0),
    ("Full Structural (0.5L + 20D)", 0.5, 20.0)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _ in ABLATION_CONFIGS}
results_bone = {name: [] for name, _, _ in ABLATION_CONFIGS}

print("\n" + "=" * 80)
print(" EXECUTING STRUCTURAL LOSS ABLATION (3 SEEDS, DISJOINT SEGMENTS)")
print("=" * 80)

for exp_name, l_len_wt, l_dir_wt in ABLATION_CONFIGS:
    print(f"\n>>> Configuration: {exp_name} <<<")
    loss_module = LossEvaluator(lambda_len=l_len_wt, lambda_dir=l_dir_wt)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=1, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_bone_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            total_pos_term, total_len_term, total_dir_term = 0.0, 0.0, 0.0
            num_batches = 0

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                loss, pos_val, len_val, dir_val = loss_module(pred, y_batch)
                loss.backward()
                optimizer.step()

                total_pos_term += pos_val
                total_len_term += len_val
                total_dir_term += dir_val
                num_batches += 1

            # Validation evaluation
            model.eval()
            total_err, total_bone_err, total_pts = 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

                    b_pred = torch.norm(pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES], dim=-1)
                    b_gt = torch.norm(y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES], dim=-1)
                    total_bone_err += torch.abs(b_pred - b_gt).sum().item()

            val_mpjpe = total_err / total_pts
            val_bone_err = total_bone_err / (len(val_indices) * len(BONE_PAIRS))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_bone_err = val_bone_err

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone[exp_name].append(best_val_bone_err)
        print(f"  [Seed {seed:<3}] Val MPJPE: {best_val_mpjpe:6.2f} mm | Bone Error: {best_val_bone_err:5.2f} mm | Final Train Terms: [Pos: {total_pos_term/num_batches:.1f}mm, Len: {total_len_term/num_batches:.1f}mm, Dir: {total_dir_term/num_batches:.4f}]")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 4. FINAL COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" STRUCTURAL LOSS ABLATION RESULTS (3-SPLIT SEGMENT DISJOINT)")
print("=" * 80)
print(f"{'Loss Formulation':<30} | {'Mean MPJPE (mm)':<18} | {'Bone Length Error (mm)':<22}")
print("-" * 80)
for exp_name, _, _ in ABLATION_CONFIGS:
    m_p = np.mean(results_mpjpe[exp_name])
    s_p = np.std(results_mpjpe[exp_name])
    m_b = np.mean(results_bone[exp_name])
    s_b = np.std(results_bone[exp_name])
    print(f"{exp_name:<30} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_b:5.2f} ± {s_b:4.2f} mm")
print("=" * 80)

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

# 21-Joint Topology & 20 Directed Bones
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
TARGET_W, TARGET_H = 256, 192

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

# GPU Sobel Extractor
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

# Data Verification / In-Memory Staging
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into VRAM...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()
    print(f"Staged {len(vram_images):,} frames into VRAM.")
else:
    print(f"Reusing {len(vram_images):,} frames in VRAM.")

# -------------------------------------------------------------------------
# KINEMATIC TREE RECONSTRUCTION LAYER
# -------------------------------------------------------------------------
def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    """
    Assembles 21 joint positions from 20 directed bone vectors.
    bone_vectors: (B, 20, 3)
    Output: (B, 21, 3) with root joint 0 = (0, 0, 0)
    """
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)

    # Recursive propagation following BONE_PAIRS order
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]

    return joints

# -------------------------------------------------------------------------
# BACKBONE & PARAMETERIZED HEADS
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Head A: Cartesian Coordinates (Baseline)
class CartesianHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.head = nn.Linear(512, 63)

    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat).view(-1, 21, 3)

# Head B: Directed Bone Vectors
class BoneVectorHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.head = nn.Linear(512, NUM_BONES * 3)

    def forward(self, x):
        feat = self.backbone(x)
        bone_vecs = self.head(feat).view(-1, NUM_BONES, 3)
        return reconstruct_joints_from_bones(bone_vecs)

# Head C: Decoupled Direction + Explicit Length
class DecoupledKinematicHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.len_head = nn.Linear(512, NUM_BONES)

    def forward(self, x):
        feat = self.backbone(x)

        # Directions on S² unit sphere
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = dir_raw / (torch.norm(dir_raw, dim=-1, keepdim=True) + 1e-7)

        # Scalar lengths guaranteed positive (Softplus initialized near ~30mm)
        lengths = F.softplus(self.len_head(feat)).unsqueeze(-1)

        bone_vecs = u_dirs * lengths
        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# THREE-METRIC EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        # 1. MPJPE (mm)
        l_pos = torch.norm(pred - target, dim=-1).mean()

        # Bone representations
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # 2. Bone Length Error (mm)
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # 3. Bone Directional Cosine Loss (for backprop)
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_pred * u_gt).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        # Bone Direction Angular Error in Degrees (for evaluation)
        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)

        # Total Structural Loss (0.5 L_len + 20.0 L_dir)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir

        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# EXPERIMENTAL MATRIX: 3 HEAD ARCHITECTURES ACROSS 3 SPLITS
# -------------------------------------------------------------------------
HEAD_CONFIGS = [
    ("Cartesian XYZ (Baseline)",   CartesianHeadModel),
    ("Directed Bone Vectors",      BoneVectorHeadModel),
    ("Decoupled Dir + Length",     DecoupledKinematicHeadModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in HEAD_CONFIGS}
results_bone_len = {name: [] for name, _ in HEAD_CONFIGS}
results_bone_dir = {name: [] for name, _ in HEAD_CONFIGS}

print("\n" + "=" * 80)
print(" BENCHMARKING KINEMATIC DECODER HEADS (3 SEEDS, SEGMENT-DISJOINT)")
print("=" * 80)

for exp_name, ModelCls in HEAD_CONFIGS:
    print(f"\n>>> Model Architecture: {exp_name} <<<")

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_len_err = 0.0
        best_val_deg_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation across 3 Metrics
            model.eval()
            total_pjpe, total_len_err, total_deg_err, total_pts = 0.0, 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch)

                    b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
                    b_gt = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Length Error (mm)
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    total_len_err += torch.abs(len_p - len_g).sum().item()

                    # Bone Direction Error (Degrees)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    total_deg_err += torch.acos(cos_s).sum().item()

            val_mpjpe = total_pjpe / total_pts
            val_len_err = total_len_err / (len(val_indices) * NUM_BONES)
            val_deg_err = (total_deg_err / (len(val_indices) * NUM_BONES)) * (180.0 / np.pi)

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_len_err = val_len_err
                best_val_deg_err = val_deg_err

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_len[exp_name].append(best_val_len_err)
        results_bone_dir[exp_name].append(best_val_deg_err)
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Length Error: {best_val_len_err:5.2f} mm | Direction Error: {best_val_deg_err:5.2f}°")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# FINAL MULTI-METRIC PERFORMANCE AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" KINEMATIC DECODER BENCHMARK: 3-SPLIT SEGMENT-DISJOINT GENERALIZATION")
print("=" * 90)
print(f"{'Head Parameterization':<28} | {'MPJPE (mm)':<18} | {'Bone Length (mm)':<18} | {'Bone Direction (deg)':<20}")
print("-" * 90)
for exp_name, _ in HEAD_CONFIGS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_l, s_l = np.mean(results_bone_len[exp_name]), np.std(results_bone_len[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<28} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_l:5.2f} ± {s_l:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

In [ ]:
# @title
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert 'vram_images' in globals(), "Dataset tensors missing from memory."

# Canonical 20-Bone Names
BONE_NAMES = [
    # Thumb
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    # Index
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    # Middle
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    # Ring
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    # Pinky
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# 1. Compute Dataset-Level Empirical Bone Statistics
with torch.no_grad():
    all_targets = vram_targets.view(-1, 21, 3)
    b_all = all_targets[:, CHILD_INDICES] - all_targets[:, PARENT_INDICES]
    lens_all = torch.norm(b_all, dim=-1) # (N, 20)

    gt_mean_lens = lens_all.mean(dim=0).cpu().numpy()
    gt_std_lens = lens_all.std(dim=0).cpu().numpy()

# 2. Calibrated Decoupled Model with Analytical Inverse-Softplus Bias
class CalibratedDecoupledModel(nn.Module):
    def __init__(self, init_lengths: np.ndarray):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.len_head = nn.Linear(512, NUM_BONES)

        # Analytical Inverse-Softplus initialization: bias = ln(exp(mean) - 1)
        inv_softplus = np.log(np.exp(init_lengths) - 1.0)
        self.len_head.bias.data = torch.tensor(inv_softplus, dtype=torch.float32)
        self.len_head.weight.data.zero_()  # Zero-init weights: start exactly at mean

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = dir_raw / (torch.norm(dir_raw, dim=-1, keepdim=True) + 1e-7)
        lengths = F.softplus(self.len_head(feat))

        bone_vecs = u_dirs * lengths.unsqueeze(-1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, lengths

# 3. Train on Seed 42 Split to Inspect Learned Length Behavior
torch.manual_seed(42)
np.random.seed(42)
shuffled_segs = np.random.permutation(unique_segments)
split_idx = int(0.8 * len(shuffled_segs))
train_segs, val_segs = set(shuffled_segs[:split_idx]), set(shuffled_segs[split_idx:])
train_idx = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
val_idx = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

model = CalibratedDecoupledModel(gt_mean_lens).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, 16):
    model.train()
    perm = train_idx[torch.randperm(len(train_idx))]
    for i in range(0, len(perm), 64):
        b = perm[i:i + 64]
        raw_rgb = vram_images[b].float()
        y_batch = vram_targets[b].view(-1, 21, 3)

        x_batch = gpu_extractor(raw_rgb)
        optimizer.zero_grad()
        pred_joints, _ = model(x_batch)
        loss, _, _, _ = eval_engine(pred_joints, y_batch)
        loss.backward()
        optimizer.step()

# 4. Bone-by-Bone Audit on Held-out Validation Segments
model.eval()
val_pred_lens = []
val_gt_lens = []

with torch.no_grad():
    for i in range(0, len(val_idx), 64):
        b = val_idx[i:i + 64]
        raw_rgb = vram_images[b].float()
        y_batch = vram_targets[b].view(-1, 21, 3)

        x_batch = gpu_extractor(raw_rgb)
        _, pred_l = model(x_batch)

        gt_b = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]
        gt_l = torch.norm(gt_b, dim=-1)

        val_pred_lens.append(pred_l.cpu())
        val_gt_lens.append(gt_l.cpu())

val_pred_lens = torch.cat(val_pred_lens, dim=0).numpy()
val_gt_lens = torch.cat(val_gt_lens, dim=0).numpy()

# 5. Output Tabular Breakdown
print("\n" + "=" * 90)
print(f"{'Bone Identifier':<24} | {'GT μ':<7} | {'Pred μ':<7} | {'GT σ':<6} | {'Pred σ':<6} | {'MAE (mm)':<8}")
print("-" * 90)
for k in range(20):
    gt_u = val_gt_lens[:, k].mean()
    pr_u = val_pred_lens[:, k].mean()
    gt_s = val_gt_lens[:, k].std()
    pr_s = val_pred_lens[:, k].std()
    mae = np.abs(val_pred_lens[:, k] - val_gt_lens[:, k]).mean()
    print(f"{BONE_NAMES[k]:<24} | {gt_u:6.2f} | {pr_u:6.2f} | {gt_s:5.2f} | {pr_s:5.2f} | {mae:6.2f}")
print("=" * 90)

In [ ]:
# @title
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert 'vram_images' in globals(), "Dataset tensors missing from memory."

# -------------------------------------------------------------------------
# 1. TOPOLOGY & EMPIRICAL CANONICAL BONE LENGTHS
# -------------------------------------------------------------------------
BONE_NAMES = [
    # Thumb
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    # Index
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    # Middle
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    # Ring
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    # Pinky
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

CANONICAL_LENGTHS = torch.tensor([
    38.52, 30.32, 28.92, 14.78,  # Thumb
    82.04, 36.16, 23.81, 13.39,  # Index
    78.78, 40.80, 28.15, 13.27,  # Middle
    75.69, 36.32, 26.51, 13.30,  # Ring
    74.02, 26.61, 18.70, 13.72   # Pinky
], dtype=torch.float32, device=DEVICE)

# -------------------------------------------------------------------------
# 2. MODEL DEFINITIONS
# -------------------------------------------------------------------------
def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1 = d6[:, :3]
    a2 = d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

# Model A: Uncoupled Global Unit Directions on S²
class UncoupledFixedLengthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", CANONICAL_LENGTHS.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Model B: Hierarchical 6D Chaining
FINGER_CHAINS = [
    (0,  [1, 2, 3]),     # Thumb
    (4,  [5, 6, 7]),     # Index
    (8,  [9, 10, 11]),   # Middle
    (12, [13, 14, 15]),  # Ring
    (16, [17, 18, 19])   # Pinky
]

class HierarchicalFKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.metacarpal_head = nn.Linear(512, 5 * 6)
        self.relative_joint_head = nn.Linear(512, 15 * 6)
        self.register_buffer("bone_lens", CANONICAL_LENGTHS)
        self.register_buffer("rest_dir", torch.tensor([0., 0., 1.], device=DEVICE).view(1, 3, 1))

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        meta_6d = self.metacarpal_head(feat).view(-1, 5, 6)
        meta_R = rotation_6d_to_matrix(meta_6d.view(-1, 6)).view(B, 5, 3, 3)

        rel_6d = self.relative_joint_head(feat).view(-1, 15, 6)
        rel_R = rotation_6d_to_matrix(rel_6d.view(-1, 6)).view(B, 15, 3, 3)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        rel_idx = 0

        for f_idx, (root_bone_idx, child_bone_indices) in enumerate(FINGER_CHAINS):
            R_parent = meta_R[:, f_idx]
            u_root = torch.bmm(R_parent, self.rest_dir.expand(B, -1, -1)).squeeze(-1)
            bone_vecs[:, root_bone_idx] = u_root * self.bone_lens[root_bone_idx]

            for c_bone_idx in child_bone_indices:
                delta_R = rel_R[:, rel_idx]
                rel_idx += 1

                R_child = torch.bmm(R_parent, delta_R)
                u_child = torch.bmm(R_child, self.rest_dir.expand(B, -1, -1)).squeeze(-1)
                bone_vecs[:, c_bone_idx] = u_child * self.bone_lens[c_bone_idx]
                R_parent = R_child

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 3. BENCHMARK EXECUTION WITH PER-BONE DIRECTION PROFILING
# -------------------------------------------------------------------------
MODELS = [
    ("Uncoupled S² Directions", UncoupledFixedLengthModel),
    ("Hierarchical 6D Chaining", HierarchicalFKModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in MODELS}
results_bone_len = {name: [] for name, _ in MODELS}
results_bone_dir = {name: [] for name, _ in MODELS}
per_bone_dir_matrix = {name: np.zeros(20) for name, _ in MODELS}

print("=" * 85)
print(" RUNNING COMPARATIVE BENCHMARK: UNCOUPLED S² VS HIERARCHICAL 6D")
print("=" * 85)

for exp_name, ModelCls in MODELS:
    print(f"\n>>> Evaluating Architecture: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_len_err = 0.0
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_len_err, total_deg_err, total_pts = 0.0, 0.0, 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch)

                    b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
                    b_gt = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Residual Bone Length Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    total_len_err += torch.abs(len_p - len_g).sum().item()

                    # Bone Direction Error
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)

                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_len_err = total_len_err / (len(val_indices) * NUM_BONES)
            val_deg_err = (cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_len_err = val_len_err
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_len[exp_name].append(best_val_len_err)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Length Residual: {best_val_len_err:5.2f} mm | Direction Error: {best_val_deg_err:5.2f}°")

    per_bone_dir_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 4. REPORT: AGGREGATE SUMMARY & PER-BONE BREAKDOWN
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" AGGREGATE GENERALIZATION RESULTS: UNCOUPLED S² VS HIERARCHICAL 6D")
print("=" * 90)
print(f"{'Architecture':<28} | {'MPJPE (mm)':<18} | {'Length Residual':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _ in MODELS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_l, s_l = np.mean(results_bone_len[exp_name]), np.std(results_bone_len[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<28} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_l:5.2f} ± {s_l:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 90)
print(" PER-BONE DIRECTION ERROR COMPARISON (MEAN OVER 3 SPLITS in DEGREES)")
print("=" * 90)
print(f"{'Bone Identifier':<24} | {'Uncoupled S²':<15} | {'Hierarchical 6D':<17} | {'Delta':<10}")
print("-" * 90)
uncoupled_deg = per_bone_dir_matrix["Uncoupled S² Directions"]
hierarch_deg = per_bone_dir_matrix["Hierarchical 6D Chaining"]

for k in range(20):
    u_val = uncoupled_deg[k]
    h_val = hierarch_deg[k]
    delta = h_val - u_val
    sign = "+" if delta > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {u_val:6.2f}°         | {h_val:6.2f}°           | {sign}{delta:5.2f}°")
print("=" * 90)

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE & TOPOLOGY SETUP
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure the dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Bone Parent-Child Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 2. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 3. EMPIRICAL CALIBRATION OF CANONICAL SKELETON
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)  # (20,)

    # Compute empirical palm rest geometry from participant0 targets:
    # Palm frame: Z = Wrist -> Middle MCP, X = lateral towards Index, Y = Palm Normal
    p_wrist = all_kps[:, 0]
    p_idx_mcp = all_kps[:, 5]
    p_mid_mcp = all_kps[:, 9]
    p_pky_mcp = all_kps[:, 17]

    z_palm = F.normalize(p_mid_mcp - p_wrist, dim=-1)
    trans_palm = F.normalize(p_pky_mcp - p_idx_mcp, dim=-1)
    y_palm = F.normalize(torch.cross(z_palm, trans_palm, dim=-1), dim=-1)
    x_palm = F.normalize(torch.cross(y_palm, z_palm, dim=-1), dim=-1)
    R_palm_all = torch.stack([x_palm, y_palm, z_palm], dim=-1)  # (N, 3, 3)

    # Project each metacarpal into the local palm frame
    meta_kps = [all_kps[:, 1], all_kps[:, 5], all_kps[:, 9], all_kps[:, 13], all_kps[:, 17]]
    meta_dirs_local = []
    for mkp in meta_kps:
        v_meta = F.normalize(mkp - p_wrist, dim=-1)  # (N, 3)
        v_local = torch.bmm(R_palm_all.transpose(1, 2), v_meta.unsqueeze(-1)).squeeze(-1)
        meta_dirs_local.append(F.normalize(v_local.mean(dim=0), dim=-1))

    EMPIRICAL_META_DIRS = torch.stack(meta_dirs_local, dim=0)  # (5, 3)
    print("Empirical Metacarpal Rest Directions Calibrated Successfully.")

# -------------------------------------------------------------------------
# 4. GPU SOBEL ENGINE & ROTATION MODULES
# -------------------------------------------------------------------------
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1 = d6[:, :3]
    a2 = d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 5. BACKBONE & HEAD ARCHITECTURES
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Architecture 1: Uncoupled S² Directions (Canonical Baseline)
class UncoupledFixedLengthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Architecture 2: Calibrated Anatomical Forward Kinematics
class CalibratedAnatomicalFKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        # 6 (Global Palm 6D) + 15 * 3 (15 local phalangeal direction updates in palm frame)
        self.palm_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)

        self.register_buffer("bone_lens", L_CANON)
        self.register_buffer("meta_dirs_local", EMPIRICAL_META_DIRS)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 1. Global Palm Rotation R_palm ∈ SO(3)
        R_palm = rotation_6d_to_matrix(self.palm_head(feat))

        # 2. Local Phalangeal Directions (in calibrated Palm Frame)
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)

        # Rotate local directions into camera frame
        phalanx_cam_u = torch.bmm(R_palm, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Metacarpals rigidly anchored to calibrated Palm Frame
        meta_cam_u = torch.matmul(R_palm, self.meta_dirs_local.transpose(0, 1)).transpose(1, 2)  # (B, 5, 3)

        # Thumb Chain
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Finger Chains (Index, Middle, Ring, Pinky)
        finger_bone_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_offset = finger_bone_offsets[f_idx]
            p_offset = 3 + f_idx * 3
            bone_vecs[:, b_offset] = meta_cam_u[:, f_idx + 1] * self.bone_lens[b_offset]
            bone_vecs[:, b_offset + 1] = phalanx_cam_u[:, p_offset] * self.bone_lens[b_offset + 1]
            bone_vecs[:, b_offset + 2] = phalanx_cam_u[:, p_offset + 1] * self.bone_lens[b_offset + 2]
            bone_vecs[:, b_offset + 3] = phalanx_cam_u[:, p_offset + 2] * self.bone_lens[b_offset + 3]

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 6. EVALUATION MODULE & LOSS
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()

        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_pred * u_gt).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir

        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 7. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
MODELS = [
    ("Uncoupled S² (Baseline)", UncoupledFixedLengthModel),
    ("Calibrated Anatomical FK", CalibratedAnatomicalFKModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in MODELS}
results_bone_dir = {name: [] for name, _ in MODELS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in MODELS}

print("\n" + "=" * 85)
print(" BENCHMARKING CALIBRATED ANATOMICAL FORWARD KINEMATICS")
print("=" * 85)

for exp_name, ModelCls in MODELS:
    print(f"\n>>> Model Configuration: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred = model(x_batch)

                    b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
                    b_gt = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)

                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = (cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Direction Error: {best_val_deg_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 8. AUDIT & DIAGNOSTIC TABLE
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" BENCHMARK SUMMARY: UNCOUPLED S² VS CALIBRATED ANATOMICAL FK")
print("=" * 90)
print(f"{'Architecture':<32} | {'MPJPE (mm)':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _ in MODELS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<32} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 90)
print(" PER-BONE DIRECTION ERROR: UNCOUPLED S² VS CALIBRATED ANATOMICAL FK")
print("=" * 90)
print(f"{'Bone Identifier':<24} | {'Uncoupled S²':<15} | {'Calibrated FK':<17} | {'Delta':<10}")
print("-" * 90)
u_deg = per_bone_deg_matrix["Uncoupled S² (Baseline)"]
a_deg = per_bone_deg_matrix["Calibrated Anatomical FK"]

for k in range(20):
    u_val = u_deg[k]
    a_val = a_deg[k]
    delta = a_val - u_val
    sign = "+" if delta > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {u_val:6.2f}°         | {a_val:6.2f}°           | {sign}{delta:5.2f}°")
print("=" * 90)

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. GEOMETRIC UTILITIES & PALM FRAME EXTRACTION
# -------------------------------------------------------------------------
def construct_palm_frame(joints_21: torch.Tensor) -> torch.Tensor:
    """
    Computes an orthonormal coordinate system from joint positions:
    joints_21: (B, 21, 3)
    Output: R_palm (B, 3, 3) with columns [x_palm, y_palm, z_palm]
    """
    p_wrist   = joints_21[:, 0]
    p_idx_mcp = joints_21[:, 5]
    p_mid_mcp = joints_21[:, 9]
    p_pky_mcp = joints_21[:, 17]

    z_palm = F.normalize(p_mid_mcp - p_wrist, dim=-1)
    trans  = F.normalize(p_pky_mcp - p_idx_mcp, dim=-1)
    y_palm = F.normalize(torch.cross(z_palm, trans, dim=-1), dim=-1)
    x_palm = F.normalize(torch.cross(y_palm, z_palm, dim=-1), dim=-1)
    return torch.stack([x_palm, y_palm, z_palm], dim=-1)

def so3_geodesic_error(R_pred: torch.Tensor, R_gt: torch.Tensor) -> torch.Tensor:
    """Computes angular distance between two SO(3) matrices in degrees."""
    R_err = torch.bmm(R_pred.transpose(1, 2), R_gt)
    tr = R_err[:, 0, 0] + R_err[:, 1, 1] + R_err[:, 2, 2]
    cos_theta = torch.clamp((tr - 1.0) / 2.0, -1.0, 1.0)
    return torch.acos(cos_theta) * (180.0 / np.pi)

# Calibrate empirical constants
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

    R_palm_gt_all = construct_palm_frame(all_kps)
    meta_kps = [all_kps[:, 1], all_kps[:, 5], all_kps[:, 9], all_kps[:, 13], all_kps[:, 17]]
    meta_dirs_local = []
    for mkp in meta_kps:
        v_meta = F.normalize(mkp - all_kps[:, 0], dim=-1)
        v_loc = torch.bmm(R_palm_gt_all.transpose(1, 2), v_meta.unsqueeze(-1)).squeeze(-1)
        meta_dirs_local.append(F.normalize(v_loc.mean(dim=0), dim=-1))
    EMPIRICAL_META_DIRS = torch.stack(meta_dirs_local, dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. MODEL ARCHITECTURES
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Model A: Uncoupled S² Directions (Baseline)
class UncoupledFixedLengthModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        joints = reconstruct_joints_from_bones(bone_vecs)
        R_palm_implied = construct_palm_frame(joints)
        return joints, R_palm_implied

# Model B: Calibrated Anatomical FK
class CalibratedAnatomicalFKModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.palm_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)
        self.register_buffer("meta_dirs_local", EMPIRICAL_META_DIRS)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        R_palm = rotation_6d_to_matrix(self.palm_head(feat))
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_palm, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        meta_cam_u = torch.matmul(R_palm, self.meta_dirs_local.transpose(0, 1)).transpose(1, 2)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs), R_palm

# Model C: Shared Palm Frame + Independent Metacarpals (Ablation D)
class FreeMetaPalmModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder()
        self.meta_head = nn.Linear(512, 5 * 3)       # 5 independent metacarpal directions (camera S²)
        self.palm_head = nn.Linear(512, 6)           # 1 global palm orientation SO(3)
        self.phalanx_head = nn.Linear(512, 15 * 3)   # 15 phalanx directions (palm local S²)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 5 unconstrained metacarpal directions in camera space
        meta_raw = self.meta_head(feat).view(B, 5, 3)
        meta_cam_u = F.normalize(meta_raw, dim=-1, eps=1e-7)

        # Explicit palm frame
        R_palm = rotation_6d_to_matrix(self.palm_head(feat))

        # 15 phalanx directions defined locally in the palm frame
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_palm, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs), R_palm

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
MODELS = [
    ("Uncoupled S² (Baseline)",       UncoupledFixedLengthModel),
    ("Calibrated Anatomical FK",      CalibratedAnatomicalFKModel),
    ("Free Metacarpals + Palm Frame", FreeMetaPalmModel)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in MODELS}
results_bone_dir = {name: [] for name, _ in MODELS}
results_palm_err = {name: [] for name, _ in MODELS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in MODELS}

print("\n" + "=" * 95)
print(" RUNNING SYSTEMATIC FACTORIZATION ABLATION & PALM ERROR AUDIT")
print("=" * 95)

for exp_name, ModelCls in MODELS:
    print(f"\n>>> Model Configuration: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_val_palm_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)

                optimizer.zero_grad()
                pred_joints, _ = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            total_palm_deg, val_frames_count = 0.0, 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb)
                    pred_joints, pred_R_palm = model(x_batch)

                    b_pred = pred_joints[:, CHILD_INDICES] - pred_joints[:, PARENT_INDICES]
                    b_gt   = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred_joints - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)
                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)

                    # Palm SO(3) Geodesic Angular Error
                    R_palm_gt = construct_palm_frame(y_batch)
                    palm_deg = so3_geodesic_error(pred_R_palm, R_palm_gt)
                    total_palm_deg += palm_deg.sum().item()
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES)
            val_palm_err = total_palm_deg / val_frames_count

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_val_palm_err = val_palm_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        results_palm_err[exp_name].append(best_val_palm_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Dir Error: {best_val_deg_err:5.2f}° | Palm Error: {best_val_palm_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. RESULTS & DIAGNOSTIC AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 95)
print(" BENCHMARK SUMMARY: FACTORIZATION & PALM FRAME ACCURACY")
print("=" * 95)
print(f"{'Architecture':<32} | {'MPJPE (mm)':<18} | {'Dir Error (deg)':<18} | {'Palm Error (deg)':<18}")
print("-" * 95)
for exp_name, _ in MODELS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    m_pl, s_pl = np.mean(results_palm_err[exp_name]), np.std(results_palm_err[exp_name])
    print(f"{exp_name:<32} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}° | {m_pl:5.2f}° ± {s_pl:4.2f}°")
print("=" * 95)

print("\n" + "=" * 95)
print(" PER-BONE DIRECTION ERROR: UNCOUPLED S² VS FREE METACARPALS VS CALIBRATED FK")
print("=" * 95)
print(f"{'Bone Identifier':<24} | {'Uncoupled S²':<14} | {'Free Meta+Palm':<16} | {'Calibrated FK':<15} | {'Delta (Free-S²)':<16}")
print("-" * 95)
u_deg = per_bone_deg_matrix["Uncoupled S² (Baseline)"]
f_deg = per_bone_deg_matrix["Free Metacarpals + Palm Frame"]
c_deg = per_bone_deg_matrix["Calibrated Anatomical FK"]

for k in range(20):
    diff = f_deg[k] - u_deg[k]
    sign = "+" if diff > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {u_deg[k]:5.2f}°        | {f_deg[k]:5.2f}°          | {c_deg[k]:5.2f}°         | {sign}{diff:5.2f}°")
print("=" * 95)

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, mode: str = "scalar") -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)
        if mode == "scalar":
            return g_mag
        elif mode == "directional":
            return torch.cat([gx, gy, g_mag], dim=1)
        raise ValueError(f"Unknown mode: {mode}")

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. BACKBONES & HEADS
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Head Formulation 1: Uncoupled S² Directions
class UncoupledHeadModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Head Formulation 2: Latent Coordinate Factorization (LCF)
class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)       # 5 independent metacarpal directions
        self.latent_frame_head = nn.Linear(512, 6)   # 1 latent coordinate frame SO(3)
        self.phalanx_head = nn.Linear(512, 15 * 3)   # 15 phalanx directions in latent frame
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 5 unconstrained metacarpal directions in camera space
        meta_raw = self.meta_head(feat).view(B, 5, 3)
        meta_cam_u = F.normalize(meta_raw, dim=-1, eps=1e-7)

        # Latent reference frame
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))

        # 15 phalanx directions rotated out of latent frame into camera space
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. 2x2 FACTORIAL BENCHMARK HARNESS
# -------------------------------------------------------------------------
CONFIGS = [
    ("Uncoupled S² + Scalar Sobel",      UncoupledHeadModel,       1, "scalar"),
    ("LCF + Scalar Sobel",               LatentFactorizationModel, 1, "scalar"),
    ("Uncoupled S² + Directional Sobel", UncoupledHeadModel,       3, "directional"),
    ("LCF + Directional Sobel",          LatentFactorizationModel, 3, "directional")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _, _ in CONFIGS}
results_bone_dir = {name: [] for name, _, _, _ in CONFIGS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _, _, _ in CONFIGS}

print("\n" + "=" * 95)
print(" RUNNING 2x2 FACTORIAL BENCHMARK: GRADIENT DIRECTION VS LATENT FACTORIZATION")
print("=" * 95)

for exp_name, ModelCls, in_ch, grad_mode in CONFIGS:
    print(f"\n>>> Running: {exp_name} ({in_ch} Channels) <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb, mode=grad_mode)

                optimizer.zero_grad()
                pred_joints = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb, mode=grad_mode)
                    pred_joints = model(x_batch)

                    b_pred = pred_joints[:, CHILD_INDICES] - pred_joints[:, PARENT_INDICES]
                    b_gt   = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred_joints - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)
                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES)

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Dir Error: {best_val_deg_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. RESULTS & PER-BONE STAIRCASE AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 2x2 FACTORIAL BENCHMARK SUMMARY")
print("=" * 90)
print(f"{'Configuration':<36} | {'MPJPE (mm)':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _, _, _ in CONFIGS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<36} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: DISTAL STAIRCASE AUDIT (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'Unc (Scalar)':<14} | {'LCF (Scalar)':<14} | {'Unc (Direct)':<14} | {'LCF (Direct)':<14}"
print(header)
print("-" * 105)

u_sc = per_bone_deg_matrix["Uncoupled S² + Scalar Sobel"]
l_sc = per_bone_deg_matrix["LCF + Scalar Sobel"]
u_di = per_bone_deg_matrix["Uncoupled S² + Directional Sobel"]
l_di = per_bone_deg_matrix["LCF + Directional Sobel"]

for k in range(20):
    print(f"{BONE_NAMES[k]:<24} | {u_sc[k]:5.2f}°        | {l_sc[k]:5.2f}°        | {u_di[k]:5.2f}°        | {l_di[k]:5.2f}°")
print("=" * 105)

In [ ]:
# @title
import os
import gc
import csv
import json
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192

KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, mode: str = "scalar") -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)
        if mode == "scalar":
            return g_mag
        elif mode == "directional":
            return torch.cat([gx, gy, g_mag], dim=1)
        raise ValueError(f"Unknown mode: {mode}")

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. BACKBONES & HEADS
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

# Head Formulation 1: Uncoupled S² Directions
class UncoupledHeadModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.dir_head = nn.Linear(512, NUM_BONES * 3)
        self.register_buffer("bone_lens", L_CANON.unsqueeze(-1))

    def forward(self, x):
        feat = self.backbone(x)
        dir_raw = self.dir_head(feat).view(-1, NUM_BONES, 3)
        u_dirs = F.normalize(dir_raw, dim=-1, eps=1e-7)
        bone_vecs = u_dirs * self.bone_lens.unsqueeze(0)
        return reconstruct_joints_from_bones(bone_vecs)

# Head Formulation 2: Latent Coordinate Factorization (LCF)
class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)       # 5 independent metacarpal directions
        self.latent_frame_head = nn.Linear(512, 6)   # 1 latent coordinate frame SO(3)
        self.phalanx_head = nn.Linear(512, 15 * 3)   # 15 phalanx directions in latent frame
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)

        # 5 unconstrained metacarpal directions in camera space
        meta_raw = self.meta_head(feat).view(B, 5, 3)
        meta_cam_u = F.normalize(meta_raw, dim=-1, eps=1e-7)

        # Latent reference frame
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))

        # 15 phalanx directions rotated out of latent frame into camera space
        phalanx_raw = self.phalanx_head(feat).view(B, 15, 3)
        phalanx_local_u = F.normalize(phalanx_raw, dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_vecs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)

        # Thumb
        bone_vecs[:, 0] = meta_cam_u[:, 0] * self.bone_lens[0]
        bone_vecs[:, 1] = phalanx_cam_u[:, 0] * self.bone_lens[1]
        bone_vecs[:, 2] = phalanx_cam_u[:, 1] * self.bone_lens[2]
        bone_vecs[:, 3] = phalanx_cam_u[:, 2] * self.bone_lens[3]

        # 4 Fingers
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_vecs[:, b_off]     = meta_cam_u[:, f_idx + 1]   * self.bone_lens[b_off]
            bone_vecs[:, b_off + 1] = phalanx_cam_u[:, p_off]     * self.bone_lens[b_off + 1]
            bone_vecs[:, b_off + 2] = phalanx_cam_u[:, p_off + 1] * self.bone_lens[b_off + 2]
            bone_vecs[:, b_off + 3] = phalanx_cam_u[:, p_off + 2] * self.bone_lens[b_off + 3]

        return reconstruct_joints_from_bones(bone_vecs)

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. 2x2 FACTORIAL BENCHMARK HARNESS
# -------------------------------------------------------------------------
CONFIGS = [
    ("Uncoupled S² + Scalar Sobel",      UncoupledHeadModel,       1, "scalar"),
    ("LCF + Scalar Sobel",               LatentFactorizationModel, 1, "scalar"),
    ("Uncoupled S² + Directional Sobel", UncoupledHeadModel,       3, "directional"),
    ("LCF + Directional Sobel",          LatentFactorizationModel, 3, "directional")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _, _ in CONFIGS}
results_bone_dir = {name: [] for name, _, _, _ in CONFIGS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _, _, _ in CONFIGS}

print("\n" + "=" * 95)
print(" RUNNING 2x2 FACTORIAL BENCHMARK: GRADIENT DIRECTION VS LATENT FACTORIZATION")
print("=" * 95)

for exp_name, ModelCls, in_ch, grad_mode in CONFIGS:
    print(f"\n>>> Running: {exp_name} ({in_ch} Channels) <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = ModelCls(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_deg_err = 0.0
        best_per_bone_deg = None

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb, mode=grad_mode)

                optimizer.zero_grad()
                pred_joints = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

            # Validation Evaluation
            model.eval()
            total_pjpe, total_pts = 0.0, 0
            cur_bone_deg_sum = torch.zeros(20, device=DEVICE)
            val_frames_count = 0

            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb, mode=grad_mode)
                    pred_joints = model(x_batch)

                    b_pred = pred_joints[:, CHILD_INDICES] - pred_joints[:, PARENT_INDICES]
                    b_gt   = y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES]

                    # MPJPE
                    dist = torch.norm(pred_joints - y_batch, dim=-1)
                    total_pjpe += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone Direction Error
                    len_p = torch.norm(b_pred, dim=-1)
                    len_g = torch.norm(b_gt, dim=-1)
                    u_p = b_pred / (len_p.unsqueeze(-1) + 1e-7)
                    u_g = b_gt / (len_g.unsqueeze(-1) + 1e-7)
                    cos_s = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
                    deg_per_bone = torch.acos(cos_s) * (180.0 / np.pi)
                    cur_bone_deg_sum += deg_per_bone.sum(dim=0)
                    val_frames_count += b_idx.shape[0]

            val_mpjpe = total_pjpe / total_pts
            val_deg_err = cur_bone_deg_sum.sum().item() / (val_frames_count * NUM_BONES)

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_deg_err = val_deg_err
                best_per_bone_deg = (cur_bone_deg_sum / val_frames_count).cpu().numpy()

        results_mpjpe[exp_name].append(best_val_mpjpe)
        results_bone_dir[exp_name].append(best_val_deg_err)
        accum_bone_deg += best_per_bone_deg
        print(f"  [Seed {seed:<3}] MPJPE: {best_val_mpjpe:6.2f} mm | Dir Error: {best_val_deg_err:5.2f}°")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 6. RESULTS & PER-BONE STAIRCASE AUDIT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" 2x2 FACTORIAL BENCHMARK SUMMARY")
print("=" * 90)
print(f"{'Configuration':<36} | {'MPJPE (mm)':<18} | {'Direction Error (deg)':<20}")
print("-" * 90)
for exp_name, _, _, _ in CONFIGS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_bone_dir[exp_name]), np.std(results_bone_dir[exp_name])
    print(f"{exp_name:<36} | {m_p:6.2f} ± {s_p:4.2f} mm | {m_d:5.2f}° ± {s_d:4.2f}°")
print("=" * 90)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: DISTAL STAIRCASE AUDIT (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'Unc (Scalar)':<14} | {'LCF (Scalar)':<14} | {'Unc (Direct)':<14} | {'LCF (Direct)':<14}"
print(header)
print("-" * 105)

u_sc = per_bone_deg_matrix["Uncoupled S² + Scalar Sobel"]
l_sc = per_bone_deg_matrix["LCF + Scalar Sobel"]
u_di = per_bone_deg_matrix["Uncoupled S² + Directional Sobel"]
l_di = per_bone_deg_matrix["LCF + Directional Sobel"]

for k in range(20):
    print(f"{BONE_NAMES[k]:<24} | {u_sc[k]:5.2f}°        | {l_sc[k]:5.2f}°        | {u_di[k]:5.2f}°        | {l_di[k]:5.2f}°")
print("=" * 105)

In [ ]:
# @title
import os
import re
import csv
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

assert MANIFEST_PATH.exists(), f"Missing manifest at {MANIFEST_PATH}"

# Extract frame number from path (e.g., 'frame_000123.jpg' or '.../000123.jpg')
def extract_frame_id(path_str: str) -> int:
    stem = Path(path_str).stem
    nums = re.findall(r"\d+", stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Cannot extract numeric frame ID from: {path_str}")

# Group records by segment
segments_data = defaultdict(list)
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        seg = row["segment"]
        frame_id = extract_frame_id(row["image_path"])
        segments_data[seg].append({
            "frame_id": frame_id,
            "image_path": row["image_path"],
            "kp_path": row["orig_kp_right"]
        })

print("=" * 100)
print(f"{'Segment Name':<35} | {'Frames':<7} | {'ID Range':<15} | {'Gaps > 1':<9} | {'Max Gap':<8} | {'Contig. Pairs':<14}")
print("-" * 100)

total_segments = len(segments_data)
total_frames = 0
total_contiguous_pairs = 0
total_gaps = 0
segment_continuity = {}

for seg, rows in sorted(segments_data.items()):
    # Sort strictly by frame_id
    rows.sort(key=lambda r: r["frame_id"])
    frame_ids = [r["frame_id"] for r in rows]

    diffs = np.diff(frame_ids)
    gaps = np.where(diffs > 1)[0]
    num_gaps = len(gaps)
    max_gap = int(np.max(diffs)) if len(diffs) > 0 else 0
    contiguous_pairs = int(np.sum(diffs == 1))

    total_frames += len(rows)
    total_contiguous_pairs += contiguous_pairs
    total_gaps += num_gaps

    segment_continuity[seg] = {
        "rows": rows,
        "diffs": diffs,
        "contiguous_pairs": contiguous_pairs
    }

    id_range_str = f"[{frame_ids[0]}..{frame_ids[-1]}]"
    print(f"{seg:<35} | {len(rows):<7} | {id_range_str:<15} | {num_gaps:<9} | {max_gap:<8} | {contiguous_pairs:<14}")

print("=" * 100)
print(f"Summary: {total_frames:,} frames across {total_segments} segments | {total_contiguous_pairs:,} contiguous step-1 pairs | {total_gaps:,} gaps")
print("=" * 100)

# -------------------------------------------------------------------------
# Empirical Motion Dynamics (Inter-frame Ground Truth Joint Velocities)
# -------------------------------------------------------------------------
KEEP_21 = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]

def resolve_file(base_dir: Path, rel_path: str, fallback: str) -> Path:
    p = base_dir / rel_path
    if p.exists(): return p
    p2 = base_dir / fallback / Path(rel_path).name
    if p2.exists(): return p2
    raise FileNotFoundError(rel_path)

step1_velocities_root_rel = []
step1_velocities_distal = []

for seg, meta in segment_continuity.items():
    rows = meta["rows"]
    for i in range(len(rows) - 1):
        if rows[i+1]["frame_id"] - rows[i]["frame_id"] == 1:
            p1 = resolve_file(DATA_DIR, rows[i]["kp_path"], "keypoints_right")
            p2 = resolve_file(DATA_DIR, rows[i+1]["kp_path"], "keypoints_right")

            with open(p1) as f: kps1 = np.array(json.load(f), dtype=np.float32)[KEEP_21]
            with open(p2) as f: kps2 = np.array(json.load(f), dtype=np.float32)[KEEP_21]

            # Root-relative joint displacement in mm
            rel1 = kps1 - kps1[0:1]
            rel2 = kps2 - kps2[0:1]
            disp = np.linalg.norm(rel2 - rel1, axis=-1)  # (21,)

            step1_velocities_root_rel.append(disp)
            step1_velocities_distal.append(disp[[4, 8, 12, 16, 20]])  # 5 Fingertips

vel_all = np.array(step1_velocities_root_rel)
vel_distal = np.array(step1_velocities_distal)

print("\nEMPIRICAL INTER-FRAME DISPLACEMENT PROFILE (Adjacent Pairs, Step=1):")
print(f"  All 21 Joints Mean Displacement : {np.mean(vel_all):5.2f} mm/frame  (Std: {np.std(vel_all):4.2f} mm/frame, Max: {np.max(vel_all):5.2f} mm/frame)")
print(f"  Fingertip Mean Displacement     : {np.mean(vel_distal):5.2f} mm/frame  (Std: {np.std(vel_distal):4.2f} mm/frame, Max: {np.max(vel_distal):5.2f} mm/frame)")
print(f"  95th Percentile Fingertip Speed : {np.percentile(vel_distal, 95):5.2f} mm/frame")

In [ ]:
# @title
import os
import re
import csv
import json
from pathlib import Path
from collections import defaultdict
import numpy as np

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

def extract_frame_id(path_str: str) -> int:
    stem = Path(path_str).stem
    nums = re.findall(r"\d+", stem)
    return int(nums[-1])

def resolve_file(base_dir: Path, rel_path: str, fallback: str) -> Path:
    p = base_dir / rel_path
    if p.exists(): return p
    p2 = base_dir / fallback / Path(rel_path).name
    if p2.exists(): return p2
    raise FileNotFoundError(rel_path)

segments_data = defaultdict(list)
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        segments_data[row["segment"]].append({
            "frame_id": extract_frame_id(row["image_path"]),
            "kp_path": row["orig_kp_right"]
        })

KEEP_21 = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]

all_diffs = []
all_displacements = []
distal_displacements = []

for seg, rows in segments_data.items():
    rows.sort(key=lambda r: r["frame_id"])
    for i in range(len(rows) - 1):
        step = rows[i+1]["frame_id"] - rows[i]["frame_id"]
        all_diffs.append(step)

        p1 = resolve_file(DATA_DIR, rows[i]["kp_path"], "keypoints_right")
        p2 = resolve_file(DATA_DIR, rows[i+1]["kp_path"], "keypoints_right")
        with open(p1) as f: k1 = np.array(json.load(f), dtype=np.float32)[KEEP_21]
        with open(p2) as f: k2 = np.array(json.load(f), dtype=np.float32)[KEEP_21]

        rel1 = k1 - k1[0:1]
        rel2 = k2 - k2[0:1]
        disp = np.linalg.norm(rel2 - rel1, axis=-1)
        all_displacements.append(disp.mean())
        distal_displacements.append(disp[[4, 8, 12, 16, 20]].mean())

diffs_arr = np.array(all_diffs)
disp_arr = np.array(all_displacements)
dist_arr = np.array(distal_displacements)

print("=" * 80)
print(" EMPIRICAL TEMPORAL SAMPLING PROFILE")
print("=" * 80)
print(f"Total Sequence Transitions: {len(diffs_arr):,}")
print(f"Frame Delta (Δframe)      : Mean={np.mean(diffs_arr):.1f} | Median={np.median(diffs_arr):.1f} | Min={np.min(diffs_arr)} | Max={np.max(diffs_arr)}")
print(f"Δframe Percentiles        : 25%={np.percentile(diffs_arr, 25):.0f} | 50%={np.percentile(diffs_arr, 50):.0f} | 75%={np.percentile(diffs_arr, 75):.0f} | 95%={np.percentile(diffs_arr, 95):.0f}")

print("-" * 80)
print(" GROUND TRUTH JOINT MOTION ACROSS EXTRACTED STEPS")
print("-" * 80)
print(f"Mean Joint Displacement   : {np.mean(disp_arr):.2f} mm/sample (Std: {np.std(disp_arr):.2f} mm)")
print(f"Distal Fingertip Motion   : {np.mean(dist_arr):.2f} mm/sample (Std: {np.std(dist_arr):.2f} mm)")
print(f"Max Fingertip Excursion   : {np.max(dist_arr):.2f} mm/sample")
print("=" * 80)